# 推导并验证矩阵相乘的求导的公式
## 一、定义
设$A_{m×p}$，$B_{p×n}$为两个矩阵,用AB或者A@B表示一般数学中定义的矩阵乘法
矩阵相乘的结果为$C_{m×n}$,即C=AB


## 二、推导
### 2.1 符号约定
令$\mathcal{L}$表示最后的损失，为标量

如果$A_{m×n}=(a_{ij})_{m×n}$是一个m×n的矩阵,且$\mathcal{L}$是$A$的一个函数，默认$A'$同为m×n的矩阵，且$A'_{m×n}=(a'_{ij})_{m×n}$，其中 $a'_{ij} = \frac{\partial \mathcal{L}}{\partial a_{ij}}$


现有$A_{m×p}$,$B_{p×n}$和$C_{m×n} = AB$

对于任意$a'_{ij}$，由多元函数求导法则得.

$$
\begin{aligned}
a'_{ij} &= \frac{\partial \mathcal{L}}{\partial a_{ij}} \\
&= \sum_{t}^{n} \sum_{s}^{m} \frac{\partial \mathcal{L}}{\partial c_{st}} * \frac{\partial c_{st}}{\partial a_{ij}} \\
&=\sum_{t}^{n} \sum_{s}^{m} c'_{st} * \frac{\partial \sum_{k=1}^{p} a_{sk} b_{kt}}{\partial a_{ij}} \\
&=\sum_{t}^{n} \sum_{s}^{m} c'_{st} * \frac{\sum_{k=1}^{p} \partial a_{sk} b_{kt}}{\partial a_{ij}} \\
&=\sum_{t}^{n} \sum_{s}^{m} c'_{st} * \frac{\partial a_{sj} b_{jt}}{\partial a_{ij}} \\
&=\sum_{t}^{n} c'_{it} * \frac{\partial a_{ij} b_{jt}}{\partial a_{ij}} \\
&=\sum_{t}^{n} c'_{it} * b_{jt} \\
\end{aligned}
$$

有没有发现$A'$这个矩阵的元素好像都是相乘再相加？完全可以用矩阵乘法来表示，稍微观察一下就可得
$$
A'=C'B^{T}
$$

同理可得
$$
B'=A^{T}C'
$$

## 三、验证
### 3.1 方案
使用pytorch自动求导进行对比，对于矩阵运算的结果C使用模拟的计算方式求Loss并求导，并告知最后的Loss关于C的微分矩阵.
即Loss=f(C)，给定$C'$，求$A'$和$B'$

In [8]:
"""
    nabla为要实现的核心函数
    forward_1 forward_2为dummy的前向计算过程
    verify是验证过程
"""
import torch
M = 4
N = 5
P = 3

def forward_1(c):
    return (torch.sigmoid(torch.sum(c)) - float((torch.randn(1) > 0)))**2

def forward_2(c):
    p = torch.softmax(c, dim=-1)
    label = torch.zeros_like(c)
    indices = torch.randint(0, label.shape[-1], (label.shape[0],))
    label[torch.arange(label.shape[0]), indices] = 1
    return torch.mean((p - label)**2)

def nabla(a, b, grad_c):

    """
    note:
        a.shape = (M, P)
        b.shape = (P, N)
        grad_c.shape = (M, N)
    return:
        a_grad and b_grad
    """
    a_grad = torch.matmul(grad_c, b.T)
    b_grad = torch.matmul(a.T, grad_c)
    return a_grad, b_grad

def verify():
    func = [forward_1, forward_2]
    a = torch.randn(M, P, requires_grad=True)
    b = torch.randn(P, N, requires_grad=True)
    for f in func:
        a.grad = None
        b.grad = None
        c = a @ b
        c.retain_grad()
        loss = f(c)
        loss.backward()
        # 验证是否为0
        a_grad_cal, b_grad_cal = nabla(a.clone().detach(), b.clone().detach(), c.grad)
        error_a = ((a.grad - a_grad_cal)**2).sum().item()
        error_b = ((b.grad - b_grad_cal)**2).sum().item()
        assert error_a < 1e-6 and error_b < 1e-6
        print(error_a, error_b)
        print(a.grad)
        print(a_grad_cal)
        print(b.grad)
        print(b_grad_cal)
verify()



0.0 0.0
tensor([[-7.0792e-05, -2.9081e-05,  1.8606e-05],
        [-7.0792e-05, -2.9081e-05,  1.8606e-05],
        [-7.0792e-05, -2.9081e-05,  1.8606e-05],
        [-7.0792e-05, -2.9081e-05,  1.8606e-05]])
tensor([[-7.0792e-05, -2.9081e-05,  1.8606e-05],
        [-7.0792e-05, -2.9081e-05,  1.8606e-05],
        [-7.0792e-05, -2.9081e-05,  1.8606e-05],
        [-7.0792e-05, -2.9081e-05,  1.8606e-05]])
tensor([[-4.3305e-05, -4.3305e-05, -4.3305e-05, -4.3305e-05, -4.3305e-05],
        [ 6.1425e-05,  6.1425e-05,  6.1425e-05,  6.1425e-05,  6.1425e-05],
        [ 1.6185e-05,  1.6185e-05,  1.6185e-05,  1.6185e-05,  1.6185e-05]])
tensor([[-4.3305e-05, -4.3305e-05, -4.3305e-05, -4.3305e-05, -4.3305e-05],
        [ 6.1425e-05,  6.1425e-05,  6.1425e-05,  6.1425e-05,  6.1425e-05],
        [ 1.6185e-05,  1.6185e-05,  1.6185e-05,  1.6185e-05,  1.6185e-05]])
0.0 0.0
tensor([[ 1.1396e-02, -8.7846e-06,  1.1258e-02],
        [-2.1197e-02, -4.7262e-03, -3.1318e-03],
        [ 3.3118e-02,  1.5825e-02,  2.42